In [1]:
import numpy as np
import pandas as pd

In [2]:
vendor_performance = pd.read_csv("https://mentorskool-platform-uploads.s3.ap-south-1.amazonaws.com/documents/577c5727-40a6-420b-92d2-439e7fc03318_83d04ac6-cb74-4a96-a06a-e0d5442aa126_vendor_summary.csv")

In [3]:
orders = pd.read_csv("https://mentorskool-platform-uploads.s3.ap-south-1.amazonaws.com/documents/8cd26fa9-51e1-4721-a9e7-8013099a9600_83d04ac6-cb74-4a96-a06a-e0d5442aa126_order_summary.csv")

## **Task 1**

In [4]:
# If the order is delivered on time/early, then it shouldn't be considered in delay
def calculate_delay(value):
  if value <= 0:
    return 0
  else:
    return value

In [5]:
# Testing the function
orders['delay_duration'].apply(calculate_delay)

0       1.72
1       2.00
2       8.32
3       6.64
4       2.98
        ... 
4757    0.00
4758    0.00
4759    0.00
4760    0.00
4761    0.00
Name: delay_duration, Length: 4762, dtype: float64

In [6]:
# Applying the function over orders data
orders['delay_duration'] = orders['delay_duration'].apply(calculate_delay)
orders

,Unnamed: 0,order_id,ship_mode,vendor_id,delivery_duration,delay_duration
0,0,CA-2014-100006,Standard Class,VEN02,33.75,1.72
1,1,CA-2014-100090,Standard Class,VEN01,10.78,2.00
2,2,CA-2014-100293,Standard Class,VEN01,39.72,8.32
3,4,CA-2014-100363,Standard Class,VEN02,4.39,6.64
4,5,CA-2014-100391,Standard Class,VEN03,29.65,2.98
...,...,...,...,...,...,...
4757,4734,US-2017-162670,Second Class,VEN02,24.02,0.00
4758,4744,US-2017-166037,Standard Class,VEN02,10.63,0.00
4759,4755,US-2017-168613,Standard Class,VEN01,39.52,0.00
4760,4759,US-2017-169488,First Class,VEN02,2.35,0.00


In [7]:
# Check: Aggregating all data at partner level as required
orders.groupby('vendor_id').agg(avg_delivery_duration = ('delivery_duration','mean'), avg_delay_duration = ('delay_duration','mean') ).reset_index()

,vendor_id,avg_delivery_duration,avg_delay_duration
0,VEN01,19.917030,4.842866
1,VEN02,20.681264,4.674031
2,VEN03,20.201032,4.811233
3,VEN04,21.147540,14.062103


In [8]:
# Aggregating all data at partner level as required
delivery_summary = orders.groupby('vendor_id').agg(avg_delivery_duration = ('delivery_duration','mean'), avg_delivery_delay = ('delay_duration','mean') ).reset_index()

In [9]:
delivery_summary

,vendor_id,avg_delivery_duration,avg_delivery_delay
0,VEN01,19.917030,4.842866
1,VEN02,20.681264,4.674031
2,VEN03,20.201032,4.811233
3,VEN04,21.147540,14.062103


In [10]:
# Adjusting decimals as per specified format
delivery_summary['avg_delivery_duration'] = delivery_summary['avg_delivery_duration'].round(1)
delivery_summary['avg_delivery_delay'] = delivery_summary['avg_delivery_delay'].round(1)

delivery_summary

,vendor_id,avg_delivery_duration,avg_delivery_delay
0,VEN01,19.9,4.8
1,VEN02,20.7,4.7
2,VEN03,20.2,4.8
3,VEN04,21.1,14.1


In [11]:
# Joining delivery data and partner performance to provide single comprehensive view
delivery_performance = pd.merge(vendor_performance,delivery_summary, on = 'vendor_id', how ='inner')
delivery_performance

,vendor_id,vendor_name,total_orders_delivered,total_orders_delivered_late,total_orders_returned,total_order_value,avg_delivery_duration,avg_delivery_delay
0,VEN01,Velocity Logistics,963,55,56,404129,19.9,4.8
1,VEN02,Seaborne Ltd.,2801,184,170,1291118,20.7,4.7
2,VEN03,Voyage Enterprises,746,43,36,356033,20.2,4.8
3,VEN04,Johnsons Logistics,252,16,22,110467,21.1,14.1


In [12]:
# Calculating return rate and late delivery rate
delivery_performance['return%'] = (100*delivery_performance['total_orders_returned']/delivery_performance['total_orders_delivered']).round(1)
delivery_performance['delayed_delivery%'] = (100*delivery_performance['total_orders_delivered_late']/delivery_performance['total_orders_delivered']).round(1)
delivery_performance

,vendor_id,vendor_name,total_orders_delivered,total_orders_delivered_late,total_orders_returned,total_order_value,avg_delivery_duration,avg_delivery_delay,return%,delayed_delivery%
0,VEN01,Velocity Logistics,963,55,56,404129,19.9,4.8,5.8,5.7
1,VEN02,Seaborne Ltd.,2801,184,170,1291118,20.7,4.7,6.1,6.6
2,VEN03,Voyage Enterprises,746,43,36,356033,20.2,4.8,4.8,5.8
3,VEN04,Johnsons Logistics,252,16,22,110467,21.1,14.1,8.7,6.3


In [13]:
# Creating partner_summary as requested in the question
delivery_performance = delivery_performance[['vendor_id', 'vendor_name', 'return%', 'delayed_delivery%', 'avg_delivery_duration','avg_delivery_delay']]
delivery_performance

,vendor_id,vendor_name,return%,delayed_delivery%,avg_delivery_duration,avg_delivery_delay
0,VEN01,Velocity Logistics,5.8,5.7,19.9,4.8
1,VEN02,Seaborne Ltd.,6.1,6.6,20.7,4.7
2,VEN03,Voyage Enterprises,4.8,5.8,20.2,4.8
3,VEN04,Johnsons Logistics,8.7,6.3,21.1,14.1


## **Task 2**

In [14]:
# Dropping avg_delivery_duration as specified in question
vendor_rank = delivery_performance.drop('avg_delivery_duration', axis = 1)
vendor_rank

,vendor_id,vendor_name,return%,delayed_delivery%,avg_delivery_delay
0,VEN01,Velocity Logistics,5.8,5.7,4.8
1,VEN02,Seaborne Ltd.,6.1,6.6,4.7
2,VEN03,Voyage Enterprises,4.8,5.8,4.8
3,VEN04,Johnsons Logistics,8.7,6.3,14.1


In [15]:
# Ranking partners based on different metrics
vendor_rank['rank_by_return%'] = vendor_rank['return%'].rank(method = 'dense').astype(int)
vendor_rank['rank_by_delayed_delivery%'] = vendor_rank['delayed_delivery%'].rank(method = 'dense').astype(int)
vendor_rank['rank_by_average_delivery_delay'] = vendor_rank['avg_delivery_delay'].rank(method = 'dense').astype(int)
vendor_rank

,vendor_id,vendor_name,return%,delayed_delivery%,avg_delivery_delay,rank_by_return%,rank_by_delayed_delivery%,rank_by_average_delivery_delay
0,VEN01,Velocity Logistics,5.8,5.7,4.8,2,1,2
1,VEN02,Seaborne Ltd.,6.1,6.6,4.7,3,4,1
2,VEN03,Voyage Enterprises,4.8,5.8,4.8,1,2,2
3,VEN04,Johnsons Logistics,8.7,6.3,14.1,4,3,3
